# 05 Fairness fix: Can the model be fairer?
My fairness audit found the model over-flags students 25+ and men, and misses scholarship holders who drop out. Here I test 3 fixes and compare fairness vs. dropouts caught.


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix
from fairlearn.metrics import MetricFrame, false_positive_rate, false_negative_rate
from fairlearn.postprocessing import ThresholdOptimizer

df = pd.read_csv('../data/data.csv', sep=';')
df.columns = df.columns.str.strip()
df['dropout'] = (df['Target'] == 'Dropout').astype(int)

CATEGORICAL = ['Marital status', 'Application mode', 'Course', 'Previous qualification',
               'Nacionality', "Mother's qualification", "Father's qualification",
               "Mother's occupation", "Father's occupation"]
SEM1 = [c for c in df.columns if c.startswith('Curricular units 1st sem')]
SEM2 = [c for c in df.columns if c.startswith('Curricular units 2nd sem')]
enroll_features = [c for c in df.columns if c not in ['Target', 'dropout'] + SEM1 + SEM2]
features = enroll_features + SEM1
CUTOFF = 0.35

X_train, X_test, y_train, y_test = train_test_split(
    df[features], df['dropout'], test_size=0.2, stratify=df['dropout'], random_state=42)

def build(cols):
    cats = [c for c in cols if c in CATEGORICAL]
    nums = [c for c in cols if c not in CATEGORICAL]
    return Pipeline([
        ('prep', ColumnTransformer([
            ('cat', OneHotEncoder(handle_unknown='ignore'), cats),
            ('num', StandardScaler(), nums)])),
        ('model', LogisticRegression(max_iter=2000, class_weight='balanced')),
    ])

age_bins, age_labels = [0, 20, 24, 100], ['17-20', '21-24', '25+']
groups = {
    'age': pd.cut(X_test['Age at enrollment'], bins=age_bins, labels=age_labels),
    'gender': X_test['Gender'].map({0: 'Female', 1: 'Male'}),
    'scholarship': X_test['Scholarship holder'].map({0: 'No', 1: 'Yes'}),
}

**Notes:** Same data, split, and cutoff as before. `build()` makes the same logistic regression model for any list of columns, so I can remove columns and compare fairly.

In [2]:
def score(name, pred, prob=None):
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    row = {'version': name, 'caught': tp, 'missed': fn, 'false_alarms': fp,
           'roc_auc': round(roc_auc_score(y_test, prob), 3) if prob is not None else None}
    for label, g in groups.items():
        mf = MetricFrame(metrics={'fa': false_positive_rate, 'miss': false_negative_rate},
                         y_true=y_test, y_pred=pred, sensitive_features=g)
        gap = mf.difference() * 100
        row[f'{label}_false_alarm_gap'] = round(gap['fa'], 1)
        row[f'{label}_miss_gap'] = round(gap['miss'], 1)
    return row

**Notes:** For each version I measure how many dropouts it catches, plus the **gap** between groups. `mf.difference()` gives the largest difference between any two groups. For example, if false alarm rates are 38% for 25+ and 16.5% for 17-20, the gap is 21.7 points. **Smaller gaps mean fairer.**

In [3]:
rows = []

# A. Original
mA = build(features).fit(X_train, y_train)
pA = mA.predict_proba(X_test)[:, 1]
rows.append(score('A. Original', (pA >= CUTOFF).astype(int), pA))

# B. Remove gender and age
fB = [c for c in features if c not in ['Gender', 'Age at enrollment']]
mB = build(fB).fit(X_train[fB], y_train)
pB = mB.predict_proba(X_test[fB])[:, 1]
rows.append(score('B. No gender/age', (pB >= CUTOFF).astype(int), pB))

# C. Also remove age proxies
fC = [c for c in fB if c not in ['Marital status', 'Application mode', 'Previous qualification']]
mC = build(fC).fit(X_train[fC], y_train)
pC = mC.predict_proba(X_test[fC])[:, 1]
rows.append(score('C. No gender/age/proxies', (pC >= CUTOFF).astype(int), pC))

# D. Different cutoffs by age group
age_train = pd.cut(X_train['Age at enrollment'], bins=age_bins, labels=age_labels)
fair = ThresholdOptimizer(estimator=mA, constraints='equalized_odds', prefit=True,
                          predict_method='predict_proba', objective='balanced_accuracy_score')
fair.fit(X_train, y_train, sensitive_features=age_train)
pD = fair.predict(X_test, sensitive_features=groups['age'], random_state=42)
rows.append(score('D. Fair cutoffs by age', pD))

results = pd.DataFrame(rows)
results

,version,caught,missed,false_alarms,roc_auc,age_false_alarm_gap,age_miss_gap,gender_false_alarm_gap,gender_miss_gap,scholarship_false_alarm_gap,scholarship_miss_gap
0,A. Original,249,35,133,0.913,21.7,15.2,13.1,5.6,17.7,32.4
1,B. No gender/age,247,37,131,0.910,19.2,15.5,3.1,4.3,18.7,35.4
2,C. No gender/age/proxies,251,33,131,0.907,13.4,13.5,3.9,0.0,17.9,33.1
3,D. Fair cutoffs by age,215,69,76,NaN,7.6,4.9,8.0,4.9,9.4,30.5


**Takeaways:**
- **B (remove gender and age):** the gender false alarm gap dropped from 13.1 to 3.1 points, while catching almost the same number of dropouts (247 vs. 249). But the age gap barely moved (21.7 to 19.2). The model still "knew" age from other columns.
- **C (also remove age proxies):** removing Marital status, Application mode (one code literally means "Over 23 years old"), and Previous qualification cut the age false alarm gap from 21.7 to 13.4 points, and the gender miss gap to 0. It even caught 2 more dropouts (251). Accuracy barely changed (AUC 0.913 to 0.907). **This is the best trade-off.**
- **D (different cutoffs by age):** the smallest age gaps (7.6 and 4.9), but it caught 34 fewer dropouts (215 vs. 249). It also uses age directly to decide, which is harder to justify.
- **Not fixed:** the model still misses about a third more scholarship holders who drop out in every version. Scholarship holders usually stay, so the few who leave look "safe." A fix would need a separate check, like a light check-in for all scholarship holders after semester 1.

**Key lesson:** removing a sensitive column isn't enough when other columns carry the same information (proxies). You have to measure the gaps, not assume they're gone.

## Fairness fix summary
Removing gender, age, and 3 age-proxy columns (Version C) made the model fairer with no loss in dropouts caught: the age false alarm gap fell from 21.7 to 13.4 points, and the gender false alarm gap from 13.1 to 3.9, while catching 251 dropouts vs. 249. Group-specific cutoffs were the fairest by age but missed 34 more dropouts. The scholarship-holder gap remains and needs a policy fix, not a model fix.